In [1]:
"""
Week 13 - Function 1 (RL-AWARE: Multi-Armed Bandit over candidate generators)
Upgrades vs Week 12:
- Treat each candidate generator as an "arm" (anchor-local, cluster, PCA-ellipse, PCA-ridge,
  PCA-extrap, Sobol-global, safe-jump).
- Do a small "probe" pull for each arm, score it with cheap EI, then allocate the remaining
  candidate budget via an epsilon-greedy + softmax policy over learned Q-values.
- Q-learning style update on Q_arm using proxy reward = max(EI) from that arm's probe set.
- Adaptive exploration rate epsilon decays with dataset size (more exploitation late).
- Slightly exploration-aware xi adjustment during acquisition (keeps <= 6dp output).
- Output clamped to [0,1]^2 and rounded to 6 decimals.

NOTE:
- This keeps the Week 12 model (feature extractor + Bayesian linear head) unchanged.
- Only the exploration design / candidate generation + selection is RL-inspired.
"""

import math
import numpy as np
from dataclasses import dataclass

import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.optim import Adam


# ------------------------ 1. Reproducibility + Base Config ------------------------

RANDOM_SEED = 123
INPUT_DIM = 2
HIDDEN_SIZES = [64, 64]

N_EPOCHS_FEATURE = 1400
LR_FEATURE = 1e-3

N_TUNING_TRIALS = 18
N_FOLDS = 4

DUP_DECIMALS = 6

def scaled_budget(n: int) -> int:
    return int(min(60000, max(22000, 1500 * n + 1000)))


# Trust-region sampling
N_BEST_ANCHORS = 3
LOCAL_SIGMA_BASE = 0.035
LOCAL_MIX_BASE = 0.70

# Clustering config (Week 11)
K_MIN = 2
K_MAX = 5
CLUSTER_CAND_FRACTION = 0.35
BOUNDARY_FRACTION = 0.25
CENTROID_PULL = 0.65

# Week 12 PCA config
PCA_TOP_FRAC = 0.40
PCA_MIN_PTS = 6
PCA_RIDGE_FRAC = 0.18
PCA_EXTRAP_FRAC = 0.10
PCA_ORTH_NOISE_RATIO = 0.35
PCA_FOCUS_THRESH = 0.78

# Acquisition compute
N_PRED_SAMPLES_CHEAP = 128
N_PRED_SAMPLES_EXPENSIVE = 640
SHORTLIST_M = 256

TOP_K = 32
TOP_P = 0.80
TEMP_GREEDY = 0.05
TEMP_FLAT = 0.35
EI_FLAT_RATIO = 1.08
SAFE_GLOBAL_JUMP_SIGMA = 0.18

DEFAULT_PRIOR_SCALE = 1.0
DEFAULT_LR_VI = 3e-3
DEFAULT_N_STEPS_VI = 4500
DEFAULT_N_PRED_SAMPLES = 256
DEFAULT_XI = 0.0025

TOP_REPORT_K = 5


# ------------------------ Week 13 RL/MAB config ------------------------

# Small probe budget per arm (kept small vs total budget)
ARM_PROBE_FRACTION = 0.035
ARM_PROBE_MIN = 220          # ensure at least a modest sample per arm
ARM_PROBE_MAX = 1200

# Epsilon schedule: higher exploration early, low late
EPS_MAX = 0.30
EPS_MIN = 0.05
EPS_DECAY = 35.0             # larger -> slower decay

# Q-learning update for arm values (proxy reward = max cheap-EI on probe)
Q_ALPHA_MAX = 0.45
Q_ALPHA_MIN = 0.15

# Softmax temperature for arm allocation (higher -> more uniform allocation)
ARM_SOFTMAX_TEMP = 0.35

# Ensure every arm gets at least this fraction of remaining budget (keeps exploration alive)
ARM_MIN_ALLOC_FRAC = 0.06

# Additional exploration aware xi scaling (very mild; keeps stability)
XI_EXPLORATION_BOOST = 0.25


def set_all_seeds(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    pyro.set_rng_seed(seed)


# ------------------------ 2. Data ------------------------

def load_data():
    X_raw = np.array([
        [0.31940389, 0.76295937],
        [0.57432921, 0.87989810],
        [0.73102363, 0.73299988],
        [0.84035342, 0.26473161],
        [0.65011406, 0.68152635],
        [0.41043714, 0.14755430],
        [0.31269116, 0.07872278],
        [0.68341817, 0.86105746],
        [0.08250725, 0.40348751],
        [0.88388983, 0.58225397],
        [0.88389,    0.98389],
        [0.37454,    0.950713],
        [0.382224,   0.951319],
        [0.782778,   0.793329],
        [0.030500,   0.037300],
        [0.646168,   0.172681],
        [0.546683,   0.562315],
        [0.814946,   0.846025],
        [0.480813,   0.998897],
        [0.588448,   0.633938],
        [0.506683,   0.522315],
        [0.631981,   0.741874],
    ], dtype=np.float64)

    y_raw = np.array([
        1.32267704e-79,
        1.03307824e-46,
        7.71087511e-16,
        3.34177101e-124,
        -3.60606264e-03,
        -2.15924904e-54,
        -2.08909327e-91,
        2.53500115e-40,
        3.60677119e-81,
        6.22985647e-48,
        9.59033053e-135,
        -1.56227724e-117,
        -4.77166224e-115,
        7.535209723645751e-36,
        1.6357533426693436e-209,
        6.327028545366271e-79,
        3.45699404834516e-08,
        -5.636591868482337e-58,
        -3.898697523271252e-111,
        -0.004682506450114445,
        4.645312412507593e-13,
        -4.110450208325289e-10
    ], dtype=np.float64)

    assert X_raw.shape[1] == INPUT_DIM
    return X_raw, y_raw


# ------------------------ 3. Signed-log transform for y ------------------------

def signed_log_transform(y: np.ndarray, s: float):
    return np.sign(y) * np.log1p(np.abs(y) / s)

def signed_log_inverse(y_t: np.ndarray, s: float):
    return np.sign(y_t) * s * np.expm1(np.abs(y_t))

def compute_scale_s(y: np.ndarray):
    med = float(np.median(np.abs(y)))
    return max(med, 1e-12)


# ------------------------ 4. Feature extractor (deterministic) ------------------------

class FeatureExtractor(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            prev_dim = h
        self.net = nn.Sequential(*layers)
        self.output_dim = prev_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class DeterministicRegressor(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes):
        super().__init__()
        self.feature_extractor = FeatureExtractor(input_dim, hidden_sizes)
        self.head = nn.Linear(self.feature_extractor.output_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.feature_extractor(x)
        return self.head(feats).squeeze(-1)

def train_feature_extractor(X: torch.Tensor, y: torch.Tensor) -> FeatureExtractor:
    set_all_seeds(RANDOM_SEED)
    model = DeterministicRegressor(INPUT_DIM, HIDDEN_SIZES)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_FEATURE)
    loss_fn = nn.MSELoss()

    model.train()
    for _ in range(N_EPOCHS_FEATURE):
        optimizer.zero_grad()
        preds = model(X)
        loss = loss_fn(preds, y)
        loss.backward()
        optimizer.step()

    feature_extractor = FeatureExtractor(INPUT_DIM, HIDDEN_SIZES)
    feature_extractor.load_state_dict(model.feature_extractor.state_dict())
    return feature_extractor


# ------------------------ 5. Bayesian linear head (Pyro) ------------------------

@dataclass
class BayesianHeadConfig:
    prior_scale: float

def make_bayesian_model(feature_extractor: FeatureExtractor, cfg: BayesianHeadConfig):
    def model(x, y=None):
        pyro.module("feature_extractor", feature_extractor, update_module_params=False)
        feats = feature_extractor(x)
        H = feats.size(-1)

        weight = pyro.sample(
            "weight",
            dist.Normal(x.new_zeros(H), cfg.prior_scale * x.new_ones(H)).to_event(1)
        )
        bias = pyro.sample("bias", dist.Normal(x.new_tensor(0.0), cfg.prior_scale))
        sigma = pyro.sample("sigma", dist.HalfCauchy(x.new_tensor(1.0)))

        mean = (feats * weight).sum(dim=-1) + bias
        with pyro.plate("data", x.size(0)):
            pyro.sample("obs", dist.Normal(mean, sigma), obs=y)

    return model

def train_bayesian_head(model, X: torch.Tensor, y: torch.Tensor, lr_vi: float, n_steps_vi: int):
    pyro.clear_param_store()
    guide = AutoDiagonalNormal(model)
    optimizer = Adam({"lr": lr_vi})
    svi = SVI(model, guide, optimizer, loss=Trace_ELBO())
    for _ in range(n_steps_vi):
        svi.step(X, y)
    return guide


# ------------------------ 6. Acquisition helpers ------------------------

def normal_cdf(x: torch.Tensor) -> torch.Tensor:
    return 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))

def normal_pdf(x: torch.Tensor) -> torch.Tensor:
    return (1.0 / math.sqrt(2.0 * math.pi)) * torch.exp(-0.5 * x**2)

def compute_ei_and_pi(samples: torch.Tensor, best_y: float, xi: float):
    mean = samples.mean(dim=0)
    std = samples.std(dim=0) + 1e-9
    gamma = (mean - best_y - xi) / std
    ei = (mean - best_y - xi) * normal_cdf(gamma) + std * normal_pdf(gamma)
    ei = torch.clamp(ei, min=0.0)
    pi = normal_cdf((mean - best_y - xi) / std)
    return ei, pi, mean, std


# ------------------------ 7. Tuning (Random Search + CV) ------------------------

def mc_predictive_mean_std(model, guide, X: torch.Tensor, n_samples: int):
    predictive = Predictive(model, guide=guide, num_samples=n_samples, return_sites=("obs",))
    with torch.no_grad():
        pred = predictive(X)["obs"]
    return pred.mean(dim=0), pred.std(dim=0) + 1e-9

def cv_score(feature_extractor: FeatureExtractor,
             X_all: torch.Tensor,
             y_all: torch.Tensor,
             prior_scale: float,
             lr_vi: float,
             n_steps_vi: int,
             n_pred_samples: int):
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for train_idx, val_idx in kf.split(X_all.cpu().numpy()):
        X_tr, y_tr = X_all[train_idx], y_all[train_idx]
        X_va, y_va = X_all[val_idx], y_all[val_idx]

        cfg = BayesianHeadConfig(prior_scale=prior_scale)
        model = make_bayesian_model(feature_extractor, cfg)
        guide = train_bayesian_head(model, X_tr, y_tr, lr_vi=lr_vi, n_steps_vi=n_steps_vi)

        mu, sd = mc_predictive_mean_std(model, guide, X_va, n_samples=n_pred_samples)
        ll = (-0.5 * torch.log(2 * math.pi * sd**2) - 0.5 * ((y_va - mu) ** 2) / (sd**2)).mean()
        scores.append(ll.item())
    return float(np.mean(scores))

def random_log_uniform(rng, low, high):
    return float(np.exp(rng.uniform(np.log(low), np.log(high))))

def tune_hyperparameters(feature_extractor: FeatureExtractor, X_all: torch.Tensor, y_all: torch.Tensor):
    rng = np.random.default_rng(RANDOM_SEED)
    steps_choices = [2500, 3500, 4500, 6000]
    pred_choices = [256, 384, 512]

    best = {
        "score": -1e18,
        "prior_scale": DEFAULT_PRIOR_SCALE,
        "lr_vi": DEFAULT_LR_VI,
        "n_steps_vi": DEFAULT_N_STEPS_VI,
        "n_pred_samples": DEFAULT_N_PRED_SAMPLES,
        "xi": DEFAULT_XI,
    }

    for _ in range(N_TUNING_TRIALS):
        prior_scale = random_log_uniform(rng, 0.08, 4.0)
        lr_vi = random_log_uniform(rng, 8e-4, 8e-3)
        n_steps_vi = int(rng.choice(steps_choices))
        n_pred_samples = int(rng.choice(pred_choices))
        xi = random_log_uniform(rng, 5e-5, 8e-3)

        try:
            score = cv_score(feature_extractor, X_all, y_all,
                             prior_scale=prior_scale,
                             lr_vi=lr_vi,
                             n_steps_vi=n_steps_vi,
                             n_pred_samples=n_pred_samples)
        except Exception:
            continue

        if score > best["score"]:
            best.update({
                "score": score,
                "prior_scale": prior_scale,
                "lr_vi": lr_vi,
                "n_steps_vi": n_steps_vi,
                "n_pred_samples": n_pred_samples,
                "xi": xi,
            })
    return best


# ------------------------ 8. Utilities: dedup + coverage ------------------------

def _rounded_key(x_np: np.ndarray, decimals: int = DUP_DECIMALS):
    return tuple(np.round(x_np.astype(np.float64), decimals=decimals).tolist())

def coverage_audit(X_raw: np.ndarray):
    X = X_raw.astype(np.float64)
    D = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=-1))
    np.fill_diagonal(D, np.inf)
    nn = D.min(axis=1)
    return {
        "nn_min": float(nn.min()),
        "nn_median": float(np.median(nn)),
        "nn_mean": float(nn.mean()),
        "spread_dim0": float(X[:, 0].max() - X[:, 0].min()),
        "spread_dim1": float(X[:, 1].max() - X[:, 1].min()),
    }

def adaptive_local_mix_from_coverage(audit: dict) -> float:
    mix = LOCAL_MIX_BASE
    if audit["nn_median"] < 0.06:
        mix = max(0.55, mix - 0.10)
    if audit["nn_median"] < 0.045:
        mix = max(0.50, mix - 0.10)
    return float(mix)


# ------------------------ 9. Clustering audit (Week 11 retained) ------------------------

def kmeans_silhouette_select(X: np.ndarray, k_min: int = K_MIN, k_max: int = K_MAX, seed: int = RANDOM_SEED):
    Xf = X.astype(np.float64)
    best = {"k": None, "sil": -1e18, "km": None}
    for k in range(k_min, min(k_max, len(Xf)-1) + 1):
        try:
            km = KMeans(n_clusters=k, random_state=seed, n_init=10)
            labels = km.fit_predict(Xf)
            sil = silhouette_score(Xf, labels)
            if sil > best["sil"]:
                best.update({"k": k, "sil": float(sil), "km": km})
        except Exception:
            continue

    if best["km"] is None:
        km = KMeans(n_clusters=2, random_state=seed, n_init=10).fit(Xf)
        return km.labels_, km.cluster_centers_, 2, float("nan")

    km = best["km"]
    return km.labels_, km.cluster_centers_, int(best["k"]), float(best["sil"])

def cluster_stats(X: np.ndarray, y: np.ndarray, labels: np.ndarray, centroids: np.ndarray):
    stats = []
    for c in range(centroids.shape[0]):
        idx = np.where(labels == c)[0]
        Xc = X[idx]
        yc = y[idx]
        d = np.linalg.norm(Xc - centroids[c], axis=1)
        stats.append({
            "cluster": int(c),
            "n": int(len(idx)),
            "mean_y": float(np.mean(yc)),
            "best_y": float(np.max(yc)),
            "radius_median": float(np.median(d)) if len(d) else float("nan"),
            "radius_mean": float(np.mean(d)) if len(d) else float("nan"),
        })
    stats_sorted = sorted(stats, key=lambda r: (r["best_y"], r["mean_y"]), reverse=True)
    best_cluster = int(stats_sorted[0]["cluster"]) if len(stats_sorted) else 0
    return stats, best_cluster

def cluster_aware_sigma(base_sigma: float, promising_radius: float):
    if not np.isfinite(promising_radius):
        return base_sigma
    if promising_radius < 0.07:
        return max(0.018, base_sigma * 0.65)
    if promising_radius < 0.10:
        return max(0.022, base_sigma * 0.80)
    if promising_radius > 0.18:
        return min(0.060, base_sigma * 1.25)
    return base_sigma

def safe_global_jump(X_train_raw_np: np.ndarray, y_train_raw_np: np.ndarray):
    best_idx = int(np.argmax(y_train_raw_np))
    x_best = X_train_raw_np[best_idx]
    rng = np.random.default_rng(RANDOM_SEED + 999)
    x = x_best + SAFE_GLOBAL_JUMP_SIGMA * rng.normal(size=(INPUT_DIM,))
    x = np.clip(x, 0.0, 1.0)
    return x.astype(np.float64)


# ------------------------ 10. PCA audit + samplers ------------------------

def pca_audit_on_top(X: np.ndarray, y: np.ndarray):
    n = X.shape[0]
    m = max(PCA_MIN_PTS, int(math.ceil(PCA_TOP_FRAC * n)))
    m = min(m, n)

    top_idx = np.argsort(-y)[:m]
    Xt = X[top_idx].astype(np.float64)

    mu = Xt.mean(axis=0, keepdims=True)
    Xc = Xt - mu

    if np.linalg.norm(Xc) < 1e-12:
        comps = np.eye(INPUT_DIM, dtype=np.float64)
        evr = np.array([1.0, 0.0], dtype=np.float64)
        return comps, mu.squeeze(0), evr, top_idx

    pca = PCA(n_components=INPUT_DIM, random_state=RANDOM_SEED)
    pca.fit(Xc)
    comps = pca.components_.astype(np.float64)
    evr = pca.explained_variance_ratio_.astype(np.float64)
    return comps, mu.squeeze(0), evr, top_idx

def sample_anisotropic_gaussian(center: np.ndarray, comps: np.ndarray, sig_along: float, sig_orth: float, n: int, rng):
    z = rng.normal(size=(n, INPUT_DIM)).astype(np.float64)
    step = (z[:, [0]] * sig_along) * comps[[0], :] + (z[:, [1]] * sig_orth) * comps[[1], :]
    Xs = center.reshape(1, -1) + step
    return np.clip(Xs, 0.0, 1.0)


# ------------------------ 11. Week 13: RL-aware candidate generation ------------------------

def _arm_epsilon(n: int) -> float:
    # decays with n (more exploitation late)
    eps = EPS_MIN + (EPS_MAX - EPS_MIN) * math.exp(-n / EPS_DECAY)
    return float(np.clip(eps, EPS_MIN, EPS_MAX))

def _arm_alpha(n: int) -> float:
    # learning rate decreases with n (more stable late)
    a = Q_ALPHA_MIN + (Q_ALPHA_MAX - Q_ALPHA_MIN) * math.exp(-n / (0.65 * EPS_DECAY))
    return float(np.clip(a, Q_ALPHA_MIN, Q_ALPHA_MAX))

def _softmax(x: np.ndarray, temp: float) -> np.ndarray:
    t = max(1e-6, float(temp))
    z = (x - x.max()) / t
    e = np.exp(z)
    return e / (e.sum() + 1e-12)

def _dedup_against_train(Xcand: torch.Tensor, X_train_raw: np.ndarray, device: torch.device):
    train_keys = set(_rounded_key(x) for x in X_train_raw)
    seen = set(train_keys)
    keep = []
    Xcand_np = Xcand.detach().cpu().numpy()
    for i in range(Xcand_np.shape[0]):
        k = _rounded_key(Xcand_np[i])
        if k in seen:
            continue
        seen.add(k)
        keep.append(i)

    if len(keep) == 0:
        return torch.rand((min(4096, max(22000, 1500 * len(X_train_raw) + 1000)), INPUT_DIM), device=device)
    return Xcand[torch.tensor(keep, device=device)]

def build_candidates_week13_rl(model, guide,
                               X_train_scaled: torch.Tensor,
                               y_train_scaled: torch.Tensor,
                               X_train_raw: np.ndarray,
                               y_train_raw: np.ndarray,
                               xi_base: float,
                               device: torch.device):
    """
    MAB over generator arms.
    Returns:
      Xcand (torch.Tensor), meta dict (contains arm allocations/Q/eps, plus Week11/12 audits)
    """
    torch.manual_seed(RANDOM_SEED)
    rng = np.random.default_rng(RANDOM_SEED + 2026)
    n = len(X_train_raw)
    max_tokens = scaled_budget(n)

    # coverage -> baseline local/global split (kept)
    audit = coverage_audit(X_train_raw)
    local_mix = adaptive_local_mix_from_coverage(audit)
    n_local = int(max_tokens * local_mix)
    n_global = max_tokens - n_local

    # clustering (kept)
    labels, centroids, k_used, sil = kmeans_silhouette_select(X_train_raw)
    cstats, c_best = cluster_stats(X_train_raw, y_train_raw, labels, centroids)
    promising_radius = [r["radius_median"] for r in cstats if r["cluster"] == c_best][0]
    base_local_sigma = cluster_aware_sigma(LOCAL_SIGMA_BASE, promising_radius)
    cent_best = centroids[c_best].astype(np.float64)

    # PCA (kept)
    comps, mu_top, evr, top_idx = pca_audit_on_top(X_train_raw, y_train_raw)
    pc1_focus = float(evr[0]) >= PCA_FOCUS_THRESH
    sig_along = float(base_local_sigma * (1.10 if pc1_focus else 0.95))
    sig_orth  = float(base_local_sigma * (PCA_ORTH_NOISE_RATIO if pc1_focus else 0.55))
    pca_center = 0.55 * cent_best + 0.45 * mu_top

    # anchors
    best_idx = np.argsort(-y_train_raw)[:min(N_BEST_ANCHORS, n)]
    anchors = torch.from_numpy(X_train_raw[best_idx].astype(np.float32)).to(device)

    # define arms (each returns a tensor of candidates)
    def arm_anchor_local(m: int) -> torch.Tensor:
        if m <= 0:
            return torch.empty((0, INPUT_DIM), device=device)
        idx = torch.randint(low=0, high=anchors.size(0), size=(m,), device=device)
        noise = (0.90 * base_local_sigma) * torch.randn((m, INPUT_DIM), device=device)
        return torch.clamp(anchors[idx] + noise, 0.0, 1.0)

    def arm_cluster(m: int) -> torch.Tensor:
        if m <= 0:
            return torch.empty((0, INPUT_DIM), device=device)
        parts = []
        cent_best_t = torch.from_numpy(cent_best.astype(np.float32)).to(device)
        n_centroid_ball = int(m * (1.0 - BOUNDARY_FRACTION))
        n_boundary = m - n_centroid_ball

        if n_centroid_ball > 0:
            noise = base_local_sigma * torch.randn((n_centroid_ball, INPUT_DIM), device=device)
            parts.append(torch.clamp(cent_best_t + noise, 0.0, 1.0))

        if n_boundary > 0:
            idx_best_cluster = np.where(labels == c_best)[0]
            if len(idx_best_cluster) > 0:
                pick = rng.choice(idx_best_cluster, size=n_boundary, replace=True)
                Xpick = X_train_raw[pick].astype(np.float32)
                Xpick_t = torch.from_numpy(Xpick).to(device)
                pulled = (1.0 - CENTROID_PULL) * Xpick_t + CENTROID_PULL * cent_best_t
                noise = (0.55 * base_local_sigma) * torch.randn((n_boundary, INPUT_DIM), device=device)
                parts.append(torch.clamp(pulled + noise, 0.0, 1.0))
            else:
                noise = base_local_sigma * torch.randn((n_boundary, INPUT_DIM), device=device)
                parts.append(torch.clamp(cent_best_t + noise, 0.0, 1.0))

        # small separator sprinkle if possible (cheap diversity)
        if centroids.shape[0] >= 2 and m >= 40:
            dcent = np.linalg.norm(centroids - centroids[c_best], axis=1)
            dcent[c_best] = np.inf
            c_nn = int(np.argmin(dcent))
            mid = 0.5 * (centroids[c_best] + centroids[c_nn])
            mid_t = torch.from_numpy(mid.astype(np.float32)).to(device)
            n_sep = max(2, int(0.08 * m))
            noise = (0.75 * base_local_sigma) * torch.randn((n_sep, INPUT_DIM), device=device)
            parts.append(torch.clamp(mid_t + noise, 0.0, 1.0))

        return torch.cat(parts, dim=0) if len(parts) else torch.empty((0, INPUT_DIM), device=device)

    def arm_pca_ellipse(m: int) -> torch.Tensor:
        if m <= 0:
            return torch.empty((0, INPUT_DIM), device=device)
        Xe = sample_anisotropic_gaussian(pca_center, comps, sig_along, sig_orth, m, rng)
        return torch.from_numpy(Xe.astype(np.float32)).to(device)

    def arm_pca_ridge(m: int) -> torch.Tensor:
        if m <= 0:
            return torch.empty((0, INPUT_DIM), device=device)
        t_scale = (0.22 if pc1_focus else 0.18)
        t = rng.normal(loc=0.0, scale=t_scale, size=(m,)).astype(np.float64)
        orth = rng.normal(size=(m,)).astype(np.float64)
        Xr = (pca_center.reshape(1, -1)
              + (t[:, None] * comps[[0], :])
              + (orth[:, None] * (sig_orth * 0.85) * comps[[1], :]))
        Xr = np.clip(Xr, 0.0, 1.0)
        return torch.from_numpy(Xr.astype(np.float32)).to(device)

    def arm_pca_extrap(m: int) -> torch.Tensor:
        if m <= 0:
            return torch.empty((0, INPUT_DIM), device=device)
        Xt = X_train_raw[top_idx].astype(np.float64)
        proj = (Xt - pca_center.reshape(1, -1)) @ comps[0].reshape(-1, 1)
        p_lo, p_hi = float(np.percentile(proj, 10)), float(np.percentile(proj, 90))
        span = max(1e-6, p_hi - p_lo)

        half = m // 2
        extra_scale = (0.30 if pc1_focus else 0.22) * span
        t_hi = rng.normal(loc=p_hi + 0.20 * span, scale=extra_scale, size=(half,))
        t_lo = rng.normal(loc=p_lo - 0.20 * span, scale=extra_scale, size=(m - half,))
        t_all = np.concatenate([t_hi, t_lo], axis=0).astype(np.float64)

        orth = rng.normal(size=(m,)).astype(np.float64)
        Xx = (pca_center.reshape(1, -1)
              + (t_all[:, None] * comps[[0], :])
              + (orth[:, None] * (sig_orth * 0.75) * comps[[1], :]))
        Xx = np.clip(Xx, 0.0, 1.0)
        return torch.from_numpy(Xx.astype(np.float32)).to(device)

    def arm_sobol_global(m: int) -> torch.Tensor:
        if m <= 0:
            return torch.empty((0, INPUT_DIM), device=device)
        sob = torch.quasirandom.SobolEngine(dimension=INPUT_DIM, scramble=True, seed=RANDOM_SEED)
        return sob.draw(m).to(device)

    def arm_safe_jump(m: int) -> torch.Tensor:
        # replicate safe jump with tiny jitter (so it can contribute multiple points)
        if m <= 0:
            return torch.empty((0, INPUT_DIM), device=device)
        x0 = safe_global_jump(X_train_raw, y_train_raw)
        Xs = x0.reshape(1, -1) + 0.020 * rng.normal(size=(m, INPUT_DIM))
        Xs = np.clip(Xs, 0.0, 1.0).astype(np.float64)
        return torch.from_numpy(Xs.astype(np.float32)).to(device)

    arms = [
        ("anchor_local", arm_anchor_local),
        ("cluster",      arm_cluster),
        ("pca_ellipse",  arm_pca_ellipse),
        ("pca_ridge",    arm_pca_ridge),
        ("pca_extrap",   arm_pca_extrap),
        ("sobol",        arm_sobol_global),
        ("safe_jump",    arm_safe_jump),
    ]

    # RL parameters
    eps = _arm_epsilon(n)
    alpha = _arm_alpha(n)

    # Probe each arm (equal probe budget) then score with cheap EI
    probe_per_arm = int(np.clip(max_tokens * ARM_PROBE_FRACTION, ARM_PROBE_MIN, ARM_PROBE_MAX))
    probe_per_arm = int(min(probe_per_arm, max(64, max_tokens // (2 * len(arms)))))

    # Exploration-aware xi (very mild): increases xi slightly when eps is high
    xi_used = float(xi_base * (1.0 + XI_EXPLORATION_BOOST * eps))

    predictive_cheap = Predictive(model, guide=guide, num_samples=N_PRED_SAMPLES_CHEAP, return_sites=("obs",))
    best_y_scaled = float(y_train_scaled.max().item())

    Q = np.zeros((len(arms),), dtype=np.float64)
    probe_rewards = np.zeros_like(Q)
    probe_best_x = [None] * len(arms)

    for i, (name, fn) in enumerate(arms):
        Xp = fn(probe_per_arm)
        Xp = _dedup_against_train(Xp, X_train_raw, device=device)
        with torch.no_grad():
            samp = predictive_cheap(Xp)["obs"]
        ei, _, _, _ = compute_ei_and_pi(samp, best_y_scaled, xi=xi_used)
        r = float(torch.max(ei).item()) if ei.numel() else 0.0
        probe_rewards[i] = r
        if ei.numel():
            j = int(torch.argmax(ei).item())
            probe_best_x[i] = Xp[j].detach().cpu().numpy().astype(np.float64)

        # Q-learning style update from zero-initialised Q (single-step)
        Q[i] = (1.0 - alpha) * Q[i] + alpha * r

    # Epsilon-greedy + softmax allocation:
    # - with prob eps, allocate more uniformly (explore)
    # - else allocate by softmax(Q)
    probs_soft = _softmax(Q, temp=ARM_SOFTMAX_TEMP)

    if rng.uniform() < eps:
        probs = np.ones_like(probs_soft) / len(probs_soft)
    else:
        probs = probs_soft

    # Remaining budget after probes (we also add all probe points to final candidate set)
    probe_total = probe_per_arm * len(arms)
    remaining = max(0, max_tokens - probe_total)

    # Enforce minimum allocation per arm
    min_alloc = int(ARM_MIN_ALLOC_FRAC * remaining)
    alloc = np.array([min_alloc] * len(arms), dtype=np.int64)
    rem2 = max(0, remaining - alloc.sum())

    if rem2 > 0:
        extra = rng.multinomial(rem2, probs)
        alloc += extra

    # Generate candidates: probe sets + allocated sets
    all_parts = []
    arm_alloc_report = []
    for i, (name, fn) in enumerate(arms):
        Xp = fn(probe_per_arm)
        Xa = fn(int(alloc[i]))
        all_parts.append(Xp)
        all_parts.append(Xa)
        arm_alloc_report.append({
            "arm": name,
            "Q": float(Q[i]),
            "probe_reward": float(probe_rewards[i]),
            "alloc": int(alloc[i]),
        })

    Xcand = torch.cat(all_parts, dim=0) if len(all_parts) else torch.empty((0, INPUT_DIM), device=device)

    # Dedup vs training + internal
    Xcand = _dedup_against_train(Xcand, X_train_raw, device=device)

    meta = {
        "max_tokens": int(max_tokens),
        "eps": float(eps),
        "alpha": float(alpha),
        "xi_used": float(xi_used),
        "arm_allocations": arm_alloc_report,
        "coverage_audit": audit,
        "local_mix": float(local_mix),
        "base_local_sigma": float(base_local_sigma),
        "cluster": {
            "k_used": int(k_used),
            "silhouette": float(sil),
            "labels": labels,
            "centroids": centroids,
            "cluster_stats": cstats,
            "promising_cluster": int(c_best),
        },
        "pca": {
            "explained_variance_ratio": evr.astype(np.float64),
            "components": comps.astype(np.float64),
            "pc1_focus": bool(pc1_focus),
            "top_frac_used": float(PCA_TOP_FRAC),
            "top_count_used": int(len(top_idx)),
        },
        "pca_center": pca_center.astype(np.float64),
        "pca_sigma_along": float(sig_along),
        "pca_sigma_orth": float(sig_orth),
    }
    return Xcand, meta


# ------------------------ 12. Decoding helpers ------------------------

def adaptive_temperature_from_ei(ei: torch.Tensor) -> float:
    if ei.numel() < 10:
        return TEMP_GREEDY
    top10 = torch.topk(ei, k=10).values
    denom = float(top10.mean().item()) + 1e-12
    ratio = float(top10[0].item()) / denom
    return TEMP_FLAT if ratio < EI_FLAT_RATIO else TEMP_GREEDY

def decode_pick_from_top(ei: torch.Tensor, temperature: float, top_k: int, top_p: float):
    ei = torch.clamp(ei, min=0.0)
    if ei.numel() == 0:
        return None

    k = min(int(top_k), int(ei.numel()))
    top_vals, top_idx = torch.topk(ei, k=k)

    if float(top_vals.sum().item()) <= 1e-12:
        return int(top_idx[0].item())

    probs = top_vals / top_vals.sum()
    sorted_probs, order = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_probs, dim=0)
    m = int(torch.searchsorted(cum, torch.tensor(top_p, device=ei.device)).item()) + 1
    m = max(1, min(m, sorted_probs.numel()))

    nucleus_order = order[:m]
    nucleus_probs = sorted_probs[:m]

    T = max(1e-6, float(temperature))
    logits = torch.log(nucleus_probs + 1e-12) / T
    weights = torch.softmax(logits, dim=0)

    j = torch.multinomial(weights, num_samples=1).item()
    chosen_top_index = int(nucleus_order[j].item())
    chosen_global_index = int(top_idx[chosen_top_index].item())
    return chosen_global_index


# ------------------------ 13. Propose next point (Week 13 RL-aware) ------------------------

def propose_next_point_week13_rl(model, guide,
                                 X_train_scaled: torch.Tensor,
                                 y_train_scaled: torch.Tensor,
                                 X_train_raw_np: np.ndarray,
                                 y_train_raw_np: np.ndarray,
                                 scaler_y_transformed: StandardScaler,
                                 y_scale_s: float,
                                 xi_base: float):
    device = X_train_scaled.device

    X_candidates, meta = build_candidates_week13_rl(
        model, guide,
        X_train_scaled=X_train_scaled,
        y_train_scaled=y_train_scaled,
        X_train_raw=X_train_raw_np,
        y_train_raw=y_train_raw_np,
        xi_base=xi_base,
        device=device
    )

    # Cheap stage already used inside builder; now do cheap shortlist -> expensive refine as before
    predictive_cheap = Predictive(model, guide=guide, num_samples=N_PRED_SAMPLES_CHEAP, return_sites=("obs",))
    with torch.no_grad():
        samples_scaled_cheap = predictive_cheap(X_candidates)["obs"]

    best_y_scaled = float(y_train_scaled.max().item())
    xi_used = float(meta["xi_used"])
    ei_cheap, pi_cheap, mean_scaled_cheap, std_scaled_cheap = compute_ei_and_pi(
        samples_scaled_cheap, best_y_scaled, xi=xi_used
    )

    if ei_cheap.numel() == 0:
        x_fallback = safe_global_jump(X_train_raw_np, y_train_raw_np)
        return x_fallback, {"fallback": True, "meta": meta}

    m = min(int(SHORTLIST_M), int(ei_cheap.numel()))
    _, short_idx = torch.topk(ei_cheap, k=m)
    X_short = X_candidates[short_idx]

    predictive_exp = Predictive(model, guide=guide, num_samples=N_PRED_SAMPLES_EXPENSIVE, return_sites=("obs",))
    with torch.no_grad():
        samples_scaled_exp = predictive_exp(X_short)["obs"]

    ei, pi, mean_scaled, std_scaled = compute_ei_and_pi(samples_scaled_exp, best_y_scaled, xi=xi_used)

    temp = adaptive_temperature_from_ei(ei)
    idx_decoded = decode_pick_from_top(ei, temperature=temp, top_k=TOP_K, top_p=TOP_P)
    idx_best = int(torch.argmax(ei).item()) if idx_decoded is None else int(idx_decoded)

    next_x = X_short[idx_best].detach().cpu().numpy()
    next_x = np.clip(next_x.astype(np.float64), 0.0, 1.0)

    # flat EI safeguard (kept)
    top1 = float(torch.max(ei).item())
    top10_mean = float(torch.topk(ei, k=min(10, ei.numel())).values.mean().item())
    flat_ratio = (top1 / (top10_mean + 1e-12)) if ei.numel() >= 10 else 999.0
    if (ei.numel() >= 10) and (flat_ratio < EI_FLAT_RATIO) and (top1 < 1e-6):
        next_x = safe_global_jump(X_train_raw_np, y_train_raw_np)

    # interpretability extras (cluster assign)
    labels = meta["cluster"]["labels"]
    centroids = meta["cluster"]["centroids"]
    d_to_cent = np.linalg.norm(centroids - next_x.reshape(1, -1), axis=1)
    c_assign = int(np.argmin(d_to_cent))
    meta["cluster"]["x_next_cluster"] = c_assign
    meta["cluster"]["x_next_centroid_dist"] = float(d_to_cent[c_assign])

    # PCA projection
    pcomps = meta["pca"]["components"]
    pcenter = meta["pca_center"]
    proj = (next_x - pcenter) @ pcomps.T
    meta["pca"]["x_next_proj_pc1"] = float(proj[0])
    meta["pca"]["x_next_proj_pc2"] = float(proj[1])

    # predicted stats for report
    mean_scaled_np = mean_scaled.detach().cpu().numpy()
    std_scaled_np = std_scaled.detach().cpu().numpy()

    mean_trans = scaler_y_transformed.inverse_transform(mean_scaled_np.reshape(-1, 1)).squeeze(-1)
    mean_raw = signed_log_inverse(mean_trans, y_scale_s)

    std_trans = std_scaled_np * scaler_y_transformed.scale_[0]
    std_raw = np.abs(
        signed_log_inverse(mean_trans + std_trans, y_scale_s) - signed_log_inverse(mean_trans, y_scale_s)
    )

    best_y_trans = float(scaler_y_transformed.inverse_transform(np.array(best_y_scaled).reshape(-1, 1)).squeeze())
    best_y_raw = float(signed_log_inverse(np.array([best_y_trans]), y_scale_s).squeeze())

    short_np = X_short.detach().cpu().numpy().astype(np.float64)
    ei_np = ei.detach().cpu().numpy().astype(np.float64)
    pi_np = pi.detach().cpu().numpy().astype(np.float64)

    topk = min(TOP_REPORT_K, ei_np.shape[0])
    top_idx = np.argsort(-ei_np)[:topk]

    report_rows = []
    for j in top_idx:
        xj = np.clip(short_np[j], 0.0, 1.0)
        report_rows.append({
            "x": np.round(xj, DUP_DECIMALS).tolist(),
            "EI": float(ei_np[j]),
            "PI": float(pi_np[j]),
            "pred_mean_raw": float(mean_raw[j]),
            "pred_std_raw": float(std_raw[j]),
        })

    info = {
        "next_x": next_x,
        "best_y_raw": best_y_raw,
        "meta": meta,
        "decode_settings": {
            "temperature_used": temp,
            "top_p": TOP_P,
            "top_k": TOP_K,
            "flat_ratio": float(flat_ratio),
            "shortlist_m": int(m),
            "pred_samples_cheap": int(N_PRED_SAMPLES_CHEAP),
            "pred_samples_expensive": int(N_PRED_SAMPLES_EXPENSIVE),
            "xi_used": float(xi_used),
        },
        "top_candidates_report": report_rows
    }

    # match selected point in shortlist for predicted mean/std if possible
    key = _rounded_key(next_x)
    jmatch = None
    for j in range(short_np.shape[0]):
        if _rounded_key(short_np[j]) == key:
            jmatch = j
            break

    if jmatch is not None:
        info["next_pred_mean"] = float(mean_raw[jmatch])
        info["next_pred_std"] = float(std_raw[jmatch])
        info["next_prob_improvement"] = float(pi_np[jmatch])
    else:
        info["next_pred_mean"] = float("nan")
        info["next_pred_std"] = float("nan")
        info["next_prob_improvement"] = float("nan")

    return next_x, info


# ------------------------ 14. Main ------------------------

def main():
    set_all_seeds(RANDOM_SEED)

    X_raw, y_raw = load_data()
    print(f"Loaded X_raw shape: {X_raw.shape}, y_raw shape: {y_raw.shape}")

    ib = int(np.argmax(y_raw))
    print("\n=== INCUMBENT BEST (RAW OBSERVED) ===")
    print(f"x_best_raw: [{X_raw[ib,0]:.6f} {X_raw[ib,1]:.6f}]")
    print(f"y_best_raw: {y_raw[ib]:.6g}")

    y_s = compute_scale_s(y_raw)
    y_trans = signed_log_transform(y_raw, y_s)

    scaler_y_trans = StandardScaler()
    y_scaled_np = scaler_y_trans.fit_transform(y_trans.reshape(-1, 1)).astype(np.float32).ravel()

    device = torch.device("cpu")
    X_t = torch.from_numpy(X_raw.astype(np.float32)).to(device)
    y_t = torch.from_numpy(y_scaled_np.astype(np.float32)).to(device)

    print("\n=== PRETRAIN FEATURE EXTRACTOR ===")
    feature_extractor = train_feature_extractor(X_t, y_t)
    feature_extractor.eval()
    print("Feature extractor trained.")

    print("\n=== HYPERPARAM TUNING (Random Search + 4-fold CV) ===")
    best_hp = tune_hyperparameters(feature_extractor, X_t, y_t)

    print("Best hyperparameters found:")
    print(f"  CV score (avg log-lik): {best_hp['score']:.6f}")
    print(f"  prior_scale:    {best_hp['prior_scale']:.6g}")
    print(f"  lr_vi:          {best_hp['lr_vi']:.6g}")
    print(f"  n_steps_vi:     {best_hp['n_steps_vi']}")
    print(f"  n_pred_samples: {best_hp['n_pred_samples']}")
    print(f"  xi (base EI/PI):{best_hp['xi']:.6g}")

    print("\n=== TRAIN FINAL BAYESIAN HEAD ON ALL DATA ===")
    bayes_cfg = BayesianHeadConfig(prior_scale=best_hp["prior_scale"])
    bayes_model = make_bayesian_model(feature_extractor, bayes_cfg)
    guide = train_bayesian_head(
        bayes_model, X_t, y_t,
        lr_vi=best_hp["lr_vi"],
        n_steps_vi=best_hp["n_steps_vi"]
    )
    print("Final Bayesian head training complete.")

    print("\n=== PROPOSE NEXT QUERY POINT (Week 13 RL-aware MAB) ===")
    next_x, info = propose_next_point_week13_rl(
        bayes_model, guide,
        X_t, y_t,
        X_train_raw_np=X_raw,
        y_train_raw_np=y_raw,
        scaler_y_transformed=scaler_y_trans,
        y_scale_s=y_s,
        xi_base=best_hp["xi"],
    )

    x6 = np.round(np.array(next_x, dtype=np.float64), 6)
    x6 = np.clip(x6, 0.0, 1.0)

    print("\n=== CURRENT BEST (from observed data) ===")
    print(f"Best observed y (original scale): {info['best_y_raw']:.6g}")

    print("\n=== PROPOSED NEXT QUERY POINT ===")
    print(f"x_next (6 d.p., in [0,1]^2): [{x6[0]:.6f} {x6[1]:.6f}]")

    print("\n=== PREDICTIONS AT x_next (best-effort) ===")
    print(f"Predicted y mean (original scale): {info.get('next_pred_mean', float('nan')):.6g}")
    print(f"Predictive std (original scale):    {info.get('next_pred_std', float('nan')):.6g}")
    print(f"Prob. improvement over best:        {info.get('next_prob_improvement', float('nan')) * 100:.2f}%")

    print("\n=== RL / MAB AUDIT (Week 13) ===")
    m = info["meta"]
    print(f"epsilon={m['eps']:.3f}, alpha={m['alpha']:.3f}, xi_used={info['decode_settings']['xi_used']:.6g}")
    for row in m["arm_allocations"]:
        print(f"  arm={row['arm']:<12} Q={row['Q']:.4e} probe_reward(maxEI)={row['probe_reward']:.4e} alloc={row['alloc']}")

    print("\n=== COVERAGE AUDIT ===")
    ca = m["coverage_audit"]
    print(f"nn_min={ca['nn_min']:.4f}, nn_median={ca['nn_median']:.4f}, nn_mean={ca['nn_mean']:.4f}")
    print(f"spread_dim0={ca['spread_dim0']:.4f}, spread_dim1={ca['spread_dim1']:.4f}")
    print(f"local_mix_used={m['local_mix']:.2f}, base_local_sigma={m['base_local_sigma']:.4f}")
    print(f"pca_sigma_along={m['pca_sigma_along']:.4f}, pca_sigma_orth={m['pca_sigma_orth']:.4f}")

    print("\n=== CLUSTERING AUDIT ===")
    cmeta = m["cluster"]
    print(f"k_used={cmeta['k_used']}, silhouette={cmeta['silhouette']}")
    print(f"promising_cluster={cmeta['promising_cluster']}")
    for row in cmeta["cluster_stats"]:
        print(f"  c={row['cluster']} n={row['n']} mean_y={row['mean_y']:.4e} best_y={row['best_y']:.4e} "
              f"radius_med={row['radius_median']:.4f}")
    print(f"x_next assigned_cluster={cmeta['x_next_cluster']}, dist_to_centroid={cmeta['x_next_centroid_dist']:.4f}")

    print("\n=== PCA AUDIT ===")
    pmeta = m["pca"]
    evr = pmeta["explained_variance_ratio"]
    print(f"explained_variance_ratio: pc1={float(evr[0]):.3f}, pc2={float(evr[1]):.3f}, pc1_focus={pmeta['pc1_focus']}")
    print(f"pc1_dir={pmeta['components'][0]}, pc2_dir={pmeta['components'][1]}")
    print(f"pca_center={m['pca_center']}")
    print(f"x_next proj: pc1={pmeta['x_next_proj_pc1']:.4f}, pc2={pmeta['x_next_proj_pc2']:.4f}")

    print("\n=== TOP EI CONTENDERS (expensive stage) ===")
    for r, row in enumerate(info["top_candidates_report"], start=1):
        x = row["x"]
        print(f"{r:>2}. x=[{x[0]:.6f} {x[1]:.6f}]  EI={row['EI']:.4e}  PI={row['PI']:.3f}  "
              f"mean={row['pred_mean_raw']:.4e}  std={row['pred_std_raw']:.4e}")

    print("\n=== DECODING / ACQUISITION SETTINGS ===")
    ds = info["decode_settings"]
    print(f"temperature_used={ds['temperature_used']}, top_p={ds['top_p']}, top_k={ds['top_k']}")
    print(f"shortlist_m={ds['shortlist_m']}, pred_samples_cheap={ds['pred_samples_cheap']}, pred_samples_expensive={ds['pred_samples_expensive']}")
    print(f"flat_ratio={ds['flat_ratio']:.4f}")

    print("\nDone.")


if __name__ == "__main__":
    main()

Loaded X_raw shape: (22, 2), y_raw shape: (22,)

=== INCUMBENT BEST (RAW OBSERVED) ===
x_best_raw: [0.546683 0.562315]
y_best_raw: 3.45699e-08

=== PRETRAIN FEATURE EXTRACTOR ===
Feature extractor trained.

=== HYPERPARAM TUNING (Random Search + 4-fold CV) ===
Best hyperparameters found:
  CV score (avg log-lik): -4.871720
  prior_scale:    2.01118
  lr_vi:          0.00130874
  n_steps_vi:     3500
  n_pred_samples: 512
  xi (base EI/PI):0.00122302

=== TRAIN FINAL BAYESIAN HEAD ON ALL DATA ===
Final Bayesian head training complete.

=== PROPOSE NEXT QUERY POINT (Week 13 RL-aware MAB) ===


C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than avai


=== CURRENT BEST (from observed data) ===
Best observed y (original scale): 3.45699e-08

=== PROPOSED NEXT QUERY POINT ===
x_next (6 d.p., in [0,1]^2): [0.379812 0.817417]

=== PREDICTIONS AT x_next (best-effort) ===
Predicted y mean (original scale): -3.31376e-12
Predictive std (original scale):    5.52488e-10
Prob. improvement over best:        6.25%

=== RL / MAB AUDIT (Week 13) ===
epsilon=0.183, alpha=0.264, xi_used=0.00127908
  arm=anchor_local Q=1.7184e-02 probe_reward(maxEI)=6.5077e-02 alloc=3647
  arm=cluster      Q=1.7367e-02 probe_reward(maxEI)=6.5767e-02 alloc=3730
  arm=pca_ellipse  Q=1.7658e-02 probe_reward(maxEI)=6.6871e-02 alloc=3676
  arm=pca_ridge    Q=1.6192e-02 probe_reward(maxEI)=6.1319e-02 alloc=3638
  arm=pca_extrap   Q=1.7391e-02 probe_reward(maxEI)=6.5859e-02 alloc=3702
  arm=sobol        Q=1.6673e-02 probe_reward(maxEI)=6.3139e-02 alloc=3620
  arm=safe_jump    Q=1.9762e-02 probe_reward(maxEI)=7.4836e-02 alloc=3657

=== COVERAGE AUDIT ===
nn_min=0.0077, nn_med